In [129]:

from tespy.networks import Network
my_plant = Network()

my_plant.units.set_defaults(
    temperature="degC", pressure="bar", enthalpy="J/kg", heat="W", power="W"
)
from tespy.components import (
    CycleCloser, Compressor, Valve, SimpleHeatExchanger
)
cc = CycleCloser('cycle closer')

# heat sink
co = SimpleHeatExchanger('condenser')
# heat source
ev = SimpleHeatExchanger('evaporator')

va = Valve('expansion valve')
cp = Compressor('compressor')
# %%[sec_4]
from tespy.connections import Connection

# connections of heat pump
c1 = Connection(cc, 'out1', ev, 'in1', label='1')
c2 = Connection(ev, 'out1', cp, 'in1', label='2')
c3 = Connection(cp, 'out1', co, 'in1', label='3')
c4 = Connection(co, 'out1', va, 'in1', label='4')
c0 = Connection(va, 'out1', cc, 'in1', label='0')


import CoolProp.CoolProp as CP
fluid_name="R134a"
T_triple = CP.Props1SI("Ttriple", fluid_name)        # Triple point temperature (K)
p_triple = CP.Props1SI("ptriple", fluid_name)        # Triple point pressure (Pa)
T_critical = CP.Props1SI("T_critical", fluid_name)  # Critical temperature (K)
p_critical = CP.Props1SI("p_critical", fluid_name)  # Critical pressure (Pa)
h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, fluid_name)
T_max_K =  T_critical*0.9
p_high = min(p_critical * 0.9, 30e5) 
h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, fluid_name)
my_plant._set_p_range([p_triple, p_high])
my_plant._set_h_range([h_min,h_max])

# this line is crutial: you have to add all connections to your network
my_plant.add_conns(c1, c2, c3, c4, c0)
co.set_attr(pr=1, Q=-150000)
ev.set_attr(pr=1)
cp.set_attr(eta_s=0.95)
c2.set_attr(T=20, x=1, fluid={'R134a': 1})
c4.set_attr(T=80, x=0)
my_plant.solve(mode='design')
print(f'COP = {abs(co.Q.val) / cp.P.val}')


 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 2.68e+05   | 6 %        | 1.80e-01   | 0.00e+00   | 1.22e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 2.20e+04   | 18 %       | 1.84e-01   | 0.00e+00   | 7.66e-12   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 6.05e-09   | 100 %      | 5.06e-14   | 0.00e+00   | 7.66e-12   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 3.00e-11   | 100 %      | 1.63e-16   | 0.00e+00   | 7.66e-12   | 0.00e+00   | 0.00e+00   | 0.00e+00   
Total iterations: 4, Calculation time: 0.00 s, Iterations per second: 2312.82
COP = 3.690936852822133


In [130]:

# --- PASO 3: Liberar presiones e imponer Potencia del Compresor ---
c2.set_attr(T=20,x=1)  # Mantenemos x=1 para vapor saturado a la salida
c4.set_attr(T=None,x=0)  # Mantenemos x=0 para líquido saturado
# Asignamos la potencia deseada
COP_objetivo = 3.690936852822133
P_comp = abs(-150000) / COP_objetivo
cp.set_attr(P=P_comp)
# Segunda resolución desde el punto inicial convergido
my_plant.solve(mode='design')
print(f"--- Cálculo Exitoso Parte 2 ---")
print(f"COP obtenido = {abs(co.Q.val) / cp.P.val:.4f}")





 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 2.33e-10   | 100 %      | 6.04e-15   | 1.07e-08   | 4.50e-10   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 2.98e-10   | 100 %      | 1.75e-15   | 2.93e-09   | 1.93e-10   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 1.33e-10   | 100 %      | 2.41e-15   | 2.91e-09   | 1.06e-10   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 2.41e-10   | 100 %      | 2.85e-15   | 8.78e-09   | 1.35e-10   | 0.00e+00   | 0.00e+00   | 0.00e+00   
Total iterations: 4, Calculation time: 0.00 s, Iterations per second: 1294.64
--- Cálculo Exitoso Parte 2 ---
COP obtenido = 3.6909


In [131]:
my_plant = Network()

my_plant.units.set_defaults(
    temperature="degC", pressure="bar", enthalpy="J/kg", heat="W", power="W"
)
from tespy.components import (
    CycleCloser, Compressor, Valve, SimpleHeatExchanger
)
cc = CycleCloser('cycle closer')

# heat sink
co = SimpleHeatExchanger('condenser')
# heat source
ev = SimpleHeatExchanger('evaporator')

va = Valve('expansion valve')
cp = Compressor('compressor')
# %%[sec_4]
from tespy.connections import Connection

# connections of heat pump
c1 = Connection(cc, 'out1', ev, 'in1', label='1')
c2 = Connection(ev, 'out1', cp, 'in1', label='2')
c3 = Connection(cp, 'out1', co, 'in1', label='3')
c4 = Connection(co, 'out1', va, 'in1', label='4')
c0 = Connection(va, 'out1', cc, 'in1', label='0')


import CoolProp.CoolProp as CP
fluid_name="R134a"
T_triple = CP.Props1SI("Ttriple", fluid_name)        # Triple point temperature (K)
p_triple = CP.Props1SI("ptriple", fluid_name)        # Triple point pressure (Pa)
T_critical = CP.Props1SI("T_critical", fluid_name)  # Critical temperature (K)
p_critical = CP.Props1SI("p_critical", fluid_name)  # Critical pressure (Pa)
h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, fluid_name)
T_max_K =  T_critical*0.9
p_high = min(p_critical * 0.9, 30e5) 
h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, fluid_name)
my_plant._set_p_range([p_triple, p_high])
my_plant._set_h_range([h_min,h_max])

# this line is crutial: you have to add all connections to your network
my_plant.add_conns(c1, c2, c3, c4, c0)
ev.set_attr(pr=1, Q=150000)
co.set_attr(pr=1)
cp.set_attr(eta_s=0.95)
c2.set_attr(T=20, x=1, fluid={'R134a': 1})
c4.set_attr(T=80, x=0)
my_plant.solve(mode='design')
print(f'COP = {abs(co.Q.val) / cp.P.val}')


 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 2.29e+04   | 18 %       | 1.01e-01   | 0.00e+00   | 2.22e+04   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 7.48e-09   | 100 %      | 8.56e-14   | 0.00e+00   | 7.66e-12   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 3.00e-11   | 100 %      | 3.33e-16   | 0.00e+00   | 7.66e-12   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 7.28e-12   | 100 %      | 0.00e+00   | 0.00e+00   | 7.66e-12   | 0.00e+00   | 0.00e+00   | 0.00e+00   
Total iterations: 4, Calculation time: 0.00 s, Iterations per second: 2607.59
COP = 3.6909368528221336


In [132]:
my_plant.print_results()


##### RESULTS (Compressor) #####
+------------+----------+----------+-----------+----------+
|            |        P |       pr |        dp |    eta_s |
|------------+----------+----------+-----------+----------|
| compressor | 5.57e+04 | 4.61e+00 | -2.06e+01 | 9.50e-01 |
+------------+----------+----------+-----------+----------+
##### RESULTS (SimpleHeatExchanger) #####
+------------+-----------+----------+----------+-----------+----------+
|            |         Q |       pr |       dp |   zeta_d4 |     zeta |
|------------+-----------+----------+----------+-----------+----------|
| condenser  | -2.06e+05 | 1.00e+00 | 0.00e+00 |  0.00e+00 | 0.00e+00 |
| evaporator |  1.50e+05 | 1.00e+00 | 0.00e+00 |  0.00e+00 | 0.00e+00 |
+------------+-----------+----------+----------+-----------+----------+
##### RESULTS (CycleCloser) #####
+--------------+------------------+-------------------+
|              |   mass_deviation |   fluid_deviation |
|--------------+------------------+-----------

In [133]:
# --- PASO 3: Liberar presiones e imponer Potencia del Compresor ---
c2.set_attr(T=20,x=1)  # Mantenemos x=1 para vapor saturado a la salida
c4.set_attr(T=None,x=0)  # Mantenemos x=0 para líquido saturado
# Asignamos la potencia deseada
COP_objetivo = 3.6909368528221336
P_comp = 150000 / (COP_objetivo-1)
cp.set_attr(P=P_comp)
# Segunda resolución desde el punto inicial convergido
my_plant.solve(mode='design')
print(f"--- Cálculo Exitoso Parte 2 ---")
print(f"COP obtenido = {abs(co.Q.val) / cp.P.val:.4f}")



 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 2.35e-10   | 100 %      | 8.27e-15   | 1.02e-08   | 4.35e-10   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 3.85e-10   | 100 %      | 5.65e-15   | 1.56e-08   | 2.69e-10   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 1.85e-10   | 100 %      | 2.98e-15   | 9.40e-09   | 2.08e-10   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 3.31e-10   | 100 %      | 3.97e-15   | 1.33e-10   | 2.93e-10   | 0.00e+00   | 0.00e+00   | 0.00e+00   
Total iterations: 4, Calculation time: 0.00 s, Iterations per second: 1006.73
--- Cálculo Exitoso Parte 2 ---
COP obtenido = 3.6909


In [134]:
my_plant.print_results()


##### RESULTS (Compressor) #####
+------------+----------+----------+-----------+----------+
|            |        P |       pr |        dp |    eta_s |
|------------+----------+----------+-----------+----------|
| compressor | 5.57e+04 | 4.61e+00 | -2.06e+01 | 9.50e-01 |
+------------+----------+----------+-----------+----------+
##### RESULTS (SimpleHeatExchanger) #####
+------------+-----------+----------+----------+-----------+----------+
|            |         Q |       pr |       dp |   zeta_d4 |     zeta |
|------------+-----------+----------+----------+-----------+----------|
| condenser  | -2.06e+05 | 1.00e+00 | 0.00e+00 |  0.00e+00 | 0.00e+00 |
| evaporator |  1.50e+05 | 1.00e+00 | 0.00e+00 |  0.00e+00 | 0.00e+00 |
+------------+-----------+----------+----------+-----------+----------+
##### RESULTS (CycleCloser) #####
+--------------+------------------+-------------------+
|              |   mass_deviation |   fluid_deviation |
|--------------+------------------+-----------